In [ ]:
import pandas as pd

df = pd.read_csv("ChildGuard.csv", encoding="cp1252")
print(df.columns)
print(df.head())

Index(['text', 'actual_class', 'predicted_class', 'Age_Group'], dtype='object')
                                                text  actual_class  \
0     personally agree sailor cuteness favour cha...             0   
1       ultimate pi day history     claim ultimat...             0   
2                                        wait home               0   
3  g4  jamesbwatson want reply wp refundgracie ba...             0   
4  redirect talk dziewicza gra great poland voivo...             0   

   predicted_class Age_Group  
0                0   Unknown  
1                0   Unknown  
2                0   Unknown  
3                0   Unknown  
4                0   Unknown  


In [1]:
# ============================================================
# 0) Gerekli Kütüphaneler
# Bu bölümde projede kullanılan tüm kütüphaneler yüklenmekte
# ve veri madenciliği kapsamında hem klasik ML hem de
# Derin Öğrenme (BERT) modelleri için gerekli altyapı kurulmaktadır.
# ============================================================

print("Kütüphaneler yükleniyor...")
!pip install transformers datasets accelerate tensorflow -U > /dev/null
print("Yükleme tamamlandı.")

import pandas as pd
import numpy as np
import re
import tensorflow as tf

from sklearn.model_selection import train_test_split         # -> Train/Test ayrımı (stratified split)
from sklearn.utils import class_weight                      # -> Dengesiz veri setleri için class_weight hesabı
from sklearn.metrics import (accuracy_score,
                             precision_recall_fscore_support,
                             classification_report)

from sklearn.linear_model import LogisticRegression         # -> Klasik ML modeli (baseline)
from sklearn.feature_extraction.text import TfidfVectorizer # -> TF-IDF: Derste işlediğimiz metin madenciliği yöntemi
from scipy.sparse import hstack, csr_matrix                 # -> TF-IDF + sayısal + kategorik birleşimi için sparse matris işlemleri

from transformers import BertTokenizer, TFBertForSequenceClassification  # -> BERT Modeli


# ============================================================
# 1) Genel Ayarlar
# Bu bölümde proje içinde sık kullanılan sabitler tanımlanmıştır.
# Metin kolonunun adı, hedef değişken, yaş grubu kolonu gibi bilgiler burada tutulmaktadır.
# ============================================================

DATASET_PATH = "ChildGuard.csv"
TEXT = "text"
TARGET = "actual_class"
AGE_GROUP_COL = "Age_Group"

MAX_LEN = 128         # BERT için maksimum token uzunluğu
BATCH_SIZE = 16       # GPU bellek kullanımını optimize eden batch size

# Yaş gruplarını ayrı modeller olarak ele almak,
# veri madenciliğinde "segmentasyon" yaklaşımına örnektir.
AGE_GROUPS = {
    "Teen":       ["teen"],
    "Pre-Teen":   ["pre-teen", "preteen"],
    "Younger":    ["younger", "child"]
}


# ============================================================
# 2) Dataset Yükleme ve Temizleme
# Bu bölümde:
# - Eksik veriler temizlenmiştir (dropna)
# - Derste işlenen “metin temizleme” teknikleri uygulanmıştır
# ============================================================

print("\nDataset yükleniyor...")
df_raw = pd.read_csv(DATASET_PATH, encoding="cp1252")
print("Kolonlar:", df_raw.columns.tolist())
print("Toplam satır:", len(df_raw))

# Eksik verilerin temizlenmesi (Data Cleaning)
df_raw = df_raw.dropna(subset=[TEXT, TARGET, AGE_GROUP_COL])

# Metin temizleme fonksiyonu:
# URL, mention, emoji, özel karakter temizliği ve lower-case dönüşümü
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_raw[TEXT] = df_raw[TEXT].apply(clean_text)


# ============================================================
# 3) Tokenizer
# BERT modelleri ham metin kullanmaz, tokenizasyon gerektirir.
# Bu bölümde tokenizer bir kere indirilip encode fonksiyonu tanımlanmıştır.
# ============================================================

print("\nTokenizer indiriliyor...")
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def encode(texts):
    return tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=MAX_LEN
    )


# ============================================================
# 4) BERT Modelleri (Her Yaş Grubu Ayrı)
# Bu bölümde:
# - Yaş grubu bazlı veri segmentasyonu yapılmıştır.
# - Stratified split ile Train/Val/Test ayrımı yapılmıştır.
# - Dengesiz sınıflar için class_weight hesaplanmıştır.
# - Her yaş grubu için ayrı bir BERT modeli eğitilmiştir.
# - Threshold tuning uygulanmıştır.
# ============================================================

results = []

for group_name, patterns in AGE_GROUPS.items():
    print("\n" + "="*60)
    print(f" YAŞ GRUBU: {group_name}")
    print("="*60)

    # Yaş grubu filtreleme (regex ile)
    pattern_regex = "|".join(patterns)
    df = df_raw[df_raw[AGE_GROUP_COL].str.contains(pattern_regex, case=False, na=False)].copy()

    print(f"{group_name} için satır sayısı:", len(df))

    # Bağımlı ve bağımsız değişkenler
    X = df[TEXT].tolist()
    y = df[TARGET].astype(int).tolist()

    # Stratified split — derste öğrendiğimiz şekilde sınıf dengesi korunarak bölme
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.1, random_state=42, stratify=y_train
    )

    # Sınıf dengesizliği için class_weight — bu proje için kritik önem taşımaktadır.
    raw_weights = class_weight.compute_class_weight(
        "balanced",
        classes=np.unique(y_train),
        y=y_train
    )
    cw = dict(enumerate(raw_weights))
    print("class_weight =", cw)

    # Metinlerin BERT input formatına çevrilmesi
    train_enc = encode(X_train)
    val_enc   = encode(X_val)
    test_enc  = encode(X_test)

    # TensorFlow Dataset oluşturma
    train_ds = tf.data.Dataset.from_tensor_slices((dict(train_enc), y_train)).shuffle(2000).batch(BATCH_SIZE)
    val_ds   = tf.data.Dataset.from_tensor_slices((dict(val_enc), y_val)).batch(BATCH_SIZE)
    test_ds  = tf.data.Dataset.from_tensor_slices((dict(test_enc), y_test)).batch(BATCH_SIZE)

    # BERT modelinin indirilmesi
    print("\nBERT modeli indiriliyor...")
    model = TFBertForSequenceClassification.from_pretrained(
        "bert-base-uncased",
        num_labels=2,
        from_pt=True
    )

    # Optimizasyon ayarları
    optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5)
    loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

    model.compile(optimizer=optimizer, loss=loss, metrics=['accuracy'])

    # EarlyStopping — Overfitting'i engellemek için derste işlenen yöntem
    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=1,
        restore_best_weights=True
    )

    # Model eğitimi
    print("\nEğitim başlıyor...")
    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=4,
        class_weight=cw,
        callbacks=[early_stop],
        verbose=1
    )

    # Model test aşaması
    logits = model.predict(test_ds).logits
    probs = tf.nn.softmax(logits, axis=1).numpy()[:, 1]
    y_pred_raw = np.argmax(logits, axis=1)

    # Performans ölçümleri
    acc_raw = accuracy_score(y_test, y_pred_raw)
    p_raw, r_raw, f_raw, _ = precision_recall_fscore_support(
        y_test, y_pred_raw, average="binary"
    )

    print("\n=== RAW SONUÇLAR ===")
    print(classification_report(
        y_test,
        y_pred_raw,
        target_names=["Not Child-Targeted", "Child-Targeted"]
    ))

    # Threshold tuning — karar eşiğini optimize etmek
    thresholds = np.linspace(0.3, 0.7, 25)
    best_thr = 0.5
    best_f1 = -1
    best_acc = -1
    best_y_pred = None

    for thr in thresholds:
        y_pred_thr = (probs >= thr).astype(int)
        f1_thr = precision_recall_fscore_support(y_test, y_pred_thr, average="binary")[2]
        acc_thr = accuracy_score(y_test, y_pred_thr)

        if f1_thr > best_f1:
            best_f1 = f1_thr
            best_thr = thr
            best_acc = acc_thr
            best_y_pred = y_pred_thr

    print("\n=== TUNED SONUÇLAR ===")
    print(f"Optimal Threshold: {best_thr:.2f}")
    print(classification_report(
        y_test,
        best_y_pred,
        target_names=["Not Child-Targeted", "Child-Targeted"]
    ))

    # Sonuçları tabloya ekleme
    results.append({
        "Age Group": group_name,
        "Samples": len(df),
        "Accuracy (raw)": acc_raw,
        "F1 (raw)": f_raw,
        "Accuracy (tuned)": best_acc,
        "F1 (tuned)": best_f1,
        "Best Threshold": best_thr
    })


# ============================================================
# 5) BERT ÖZET TABLOSU
# Tüm yaş gruplarına ait performans metrikleri
# tek bir tabloya özetlenmiştir.
# ============================================================

results_df = pd.DataFrame(results)
print("\n===== BERT ÖZET TABLO (Age-Aware) =====")
print(results_df.to_string(index=False))


# ============================================================
# 6) KLASİK MODEL (TF-IDF + One-Hot + Logistic Regression)
# Bu bölüm veri madenciliği dersindeki teknikleri göstermektedir:
# - TF-IDF ile nitelik çıkarımı
# - One-Hot Encoding ile kategorik veri dönüştürme
# - Sayısal öznitelik çıkarımı
# - Logistic Regression (baseline model)
# ============================================================

print("\n\n===== KLASİK MODEL (TF-IDF + One-Hot + Logistic Regression) =====")

df_classic = df_raw.copy()

# Derste işlenen “öznitelik çıkarımı” örneği:
df_classic["text_length"] = df_classic[TEXT].str.len()
df_classic["word_count"] = df_classic[TEXT].str.split().str.len()

X_meta = df_classic[[AGE_GROUP_COL, "text_length", "word_count"]]
X_text = df_classic[TEXT]
y = df_classic[TARGET].astype(int)

# Stratified split
X_meta_train, X_meta_test, y_train, y_test, text_train, text_test = train_test_split(
    X_meta, y, X_text, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF — derste işlenen metin temsil yöntemi
vectorizer = TfidfVectorizer(max_features=5000, min_df=5)
X_train_tfidf = vectorizer.fit_transform(text_train)
X_test_tfidf = vectorizer.transform(text_test)

# One-Hot Encoding — kategorik veriyi sayısallaştırma
train_cat = pd.get_dummies(X_meta_train[AGE_GROUP_COL], prefix="AgeGroup", drop_first=True, dummy_na=True)
test_cat = pd.get_dummies(X_meta_test[AGE_GROUP_COL], prefix="AgeGroup", drop_first=True, dummy_na=True)
test_cat = test_cat.reindex(columns=train_cat.columns, fill_value=0)

# Sayısal veriler sparse matrise çevriliyor
train_num_sparse = csr_matrix(X_meta_train[["text_length", "word_count"]].values)
test_num_sparse  = csr_matrix(X_meta_test[["text_length", "word_count"]].values)

train_cat_sparse = csr_matrix(train_cat.values)
test_cat_sparse  = csr_matrix(test_cat.values)

# Tüm özelliklerin birleşimi
X_train_final = hstack([X_train_tfidf, train_num_sparse, train_cat_sparse])
X_test_final  = hstack([X_test_tfidf,  test_num_sparse,  test_cat_sparse])

# Logistic Regression — baseline model
log_reg = LogisticRegression(
    max_iter=1000,
    solver="liblinear",
    class_weight="balanced"
)

log_reg.fit(X_train_final, y_train)
y_pred = log_reg.predict(X_test_final)

acc_log = accuracy_score(y_test, y_pred)
print("\n[LOGISTIC REGRESSION - TF-IDF] Accuracy:", acc_log)
print(classification_report(
    y_test,
    y_pred,
    target_names=["Not Child-Targeted", "Child-Targeted"]
))


Kütüphaneler yükleniyor...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-text 2.19.0 requires tensorflow<2.20,>=2.19.0, but you have tensorflow 2.20.0 which is incompatible.
tf-keras 2.19.0 requires tensorflow<2.20,>=2.19, but you have tensorflow 2.20.0 which is incompatible.
tensorflow-decision-forests 1.12.0 requires tensorflow==2.19.0, but you have tensorflow 2.20.0 which is incompatible.
Yükleme tamamlandı.

Dataset yükleniyor...
Kolonlar: ['text', 'actual_class', 'predicted_class', 'Age_Group']
Toplam satır: 351877

Tokenizer indiriliyor...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...


Eğitim başlıyor...
Epoch 1/4
785/785 [==============================] - 144s 125ms/step - loss: 0.4377 - accuracy: 0.7863 - val_loss: 0.3944 - val_accuracy: 0.8179
Epoch 2/4
785/785 [==============================] - 61s 78ms/step - loss: 0.2996 - accuracy: 0.8632 - val_loss: 0.3795 - val_accuracy: 0.8301
Epoch 3/4
218/218 [==============================] - 8s 25ms/step

=== RAW SONUÇLAR ===
                    precision    recall  f1-score   support

Not Child-Targeted       0.93      0.79      0.85      2219
    Child-Targeted       0.71      0.89      0.79      1267

          accuracy                           0.83      3486
         macro avg       0.82      0.84      0.82      3486
      weighted avg       0.85      0.83      0.83      3486


=== TUNED SONUÇLAR ===
Optimal Threshold: 0.57
                    precision    recall  f1-score   support

Not Child-Targeted       0.91      0.82      0.86      2219
    Child-Targeted       0.73      0.86      0.79      1267

          a

All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Eğitim başlıyor...
Epoch 1/4
268/268 [==============================] - 89s 172ms/step - loss: 0.4965 - accuracy: 0.7551 - val_loss: 0.4372 - val_accuracy: 0.8067
Epoch 2/4
268/268 [==============================] - 31s 114ms/step - loss: 0.3293 - accuracy: 0.8574 - val_loss: 0.3799 - val_accuracy: 0.8298
Epoch 3/4
268/268 [==============================] - 26s 95ms/step - loss: 0.2133 - accuracy: 0.9178 - val_loss: 0.4377 - val_accuracy: 0.8445
Epoch 4/4
75/75 [==============================] - 5s 26ms/step

=== RAW SONUÇLAR ===
                    precision    recall  f1-score   support

Not Child-Targeted       0.91      0.87      0.89       808
    Child-Targeted       0.75      0.82      0.78       382

          accuracy                           0.86      1190
         macro avg       0.83      0.85      0.84      1190
      weighted avg       0.86      0.86      0.86      1190


=== TUNED SONUÇLAR ===
Optimal Threshold: 0.70
                    precision    recall  f1-score   

All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Eğitim başlıyor...
Epoch 1/4
1544/1544 [==============================] - 203s 104ms/step - loss: 0.3687 - accuracy: 0.8272 - val_loss: 0.3901 - val_accuracy: 0.8149
Epoch 2/4
1544/1544 [==============================] - 114s 74ms/step - loss: 0.2340 - accuracy: 0.8897 - val_loss: 0.3456 - val_accuracy: 0.8633
Epoch 3/4
1544/1544 [==============================] - 109s 71ms/step - loss: 0.1566 - accuracy: 0.9310 - val_loss: 0.2591 - val_accuracy: 0.9052
Epoch 4/4
429/429 [==============================] - 14s 26ms/step

=== RAW SONUÇLAR ===
                    precision    recall  f1-score   support

Not Child-Targeted       0.97      0.94      0.95      5985
    Child-Targeted       0.67      0.77      0.71       875

          accuracy                           0.92      6860
         macro avg       0.82      0.86      0.83      6860
      weighted avg       0.93      0.92      0.92      6860


=== TUNED SONUÇLAR ===
Optimal Threshold: 0.52
                    precision    recall  